In [ ]:
#earthquake
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. 数据加载和预处理（修改为删除异常值） ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    加载并预处理数据
    
    参数:
    - file_path: 数据文件路径
    - outlier_method: 异常值处理方法 ('remove'=删除, 'clip'=截断)
    - iqr_multiplier: IQR倍数，用于确定异常值边界（默认3倍，更宽松）
    """
    print("正在加载数据...")
    data = pd.read_csv(file_path)
    print(f"原始数据形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 检查是否存在L列
    if 'L(m)' not in data.columns:
        raise ValueError("数据中未找到'L(m)'列，请检查数据格式")
    
    # 处理缺失值
    print(f"\n缺失值统计:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"删除缺失值: {initial_shape[0] - data.shape[0]} 行")
    print(f"删除缺失值后数据形状: {data.shape}")
    
    # 保存处理前的数据用于对比
    data_before_outlier = data.copy()
    
    # 异常值检测
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\n异常值检测（使用{iqr_multiplier}倍IQR方法）:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== 删除异常值方法 ==========
        # 创建一个布尔掩码，标记所有正常值
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 找出异常值
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # 统计异常值
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\n列 '{col}':")
                print(f"  范围: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  正常值边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  异常值数量: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - 低于下界: {n_outliers_low} 个")
                print(f"    - 高于上界: {n_outliers_high} 个")
            
            # 更新掩码（只保留所有特征都正常的行）
            mask = mask & ~col_outliers
        
        # 应用掩码删除异常值
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"异常值删除总结:")
        print(f"  原始数据行数: {len(data)}")
        print(f"  删除异常值行数: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  剩余数据行数: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== 截断异常值方法（原方法） ==========
        print("\n使用截断法处理异常值...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 统计异常值数量
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\n列 '{col}':")
                print(f"  - 低于下界的值: {outliers_low} 个")
                print(f"  - 高于上界的值: {outliers_high} 个")
                print(f"  - 边界: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # 进行截断
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\n总共截断了 {modified_count} 个异常值")
    
    print(f"\n最终数据形状: {data.shape}")
    
    # 显示处理前后的统计对比
    print("\n" + "="*50)
    print("数据统计信息对比:")
    print("="*50)
    print("\n处理前统计信息:")
    print(data_before_outlier.describe().round(2))
    print("\n处理后统计信息:")
    print(data.describe().round(2))
    
    # 分离特征和目标
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. 增强的GAN模型定义 ====================
class FeatureAttention(nn.Module):
    """特征注意力机制"""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """增强的生成器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # 输入投影层
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # 隐藏层
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # 输出层
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """增强的判别器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. 优化的GAN训练类 ====================
class OptimizedGANRegressor:
    """优化的GAN回归模型"""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # 初始化网络
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # 优化器
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # 损失函数
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # 训练历史
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # 早停参数
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """计算梯度惩罚"""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """单步训练"""
        batch_size = real_x.size(0)
        
        # 动态调整标签平滑
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # 训练判别器
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # 训练生成器
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """训练模型"""
        self.generator.train()
        self.discriminator.train()
        
        # 创建数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # 训练循环
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # 学习率调度
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # 评估
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # 记录历史
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # 保存最佳模型
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # 早停
            if self.patience_counter >= patience and epoch > 100:
                print(f"\n早停触发！")
                print(f"最佳训练R2: {self.best_train_r2:.4f}")
                print(f"最佳验证R2: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # 如果训练集R2仍然很低，继续训练
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\n训练集R2仍然较低({train_r2:.4f})，继续训练...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """预测"""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """评估模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. 集成学习包装器 ====================
class OptimizedEnsembleGAN:
    """优化的集成GAN模型"""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """训练多个模型"""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"训练模型 {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # 设置不同的随机种子
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # 创建并训练模型
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # 训练模型
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # 评估
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\n模型 {i+1} 结果:")
            print(f"  训练集R2: {train_score:.4f}")
            print(f"  验证集R2: {val_score:.4f}")
        
        # 计算加权平均的权重
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\n模型权重: {self.weights}")
    
    def predict(self, X):
        """集成预测"""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """评估集成模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. 模型诊断和可视化函数 ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """诊断模型性能"""
    print("\n" + "="*50)
    print("模型诊断报告")
    print("="*50)
    
    # 训练集性能
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\n训练集性能（原始尺度）:")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # 验证集性能
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\n验证集性能（原始尺度）:")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # 诊断结论
    print("\n" + "-"*50)
    print("诊断结论:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ 异常情况：训练集欠拟合，但验证集表现良好")
            print("  可能原因：训练集包含更多噪声或异常值")
        else:
            print("⚠️ 模型存在欠拟合问题")
            print(f"  - 训练集R²较低: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ 模型存在过拟合")
        print(f"  - 训练集与验证集R²差异: {r2_diff:.4f}")
    else:
        print("✅ 模型表现良好")
        print(f"  - 训练集R²: {train_r2:.4f}")
        print(f"  - 验证集R²: {val_r2:.4f}")
        print(f"  - 差异: {r2_diff:.4f}")
    
    # 检查是否达到目标
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 目标达成！")
        print(f"  - 验证集R² >= 0.8: {val_r2:.4f}")
        print(f"  - 训练集R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ 验证集达标(R²={val_r2:.4f})，但训练集仍需改进(R²={train_r2:.4f})")
    else:
        print(f"\n❌ 未达到目标（验证集R²={val_r2:.4f} < 0.8）")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """绘制结果图表"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 处理history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. 损失曲线
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练损失曲线')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R²曲线
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R²分数变化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 训练集预测vs实际
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'训练集: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. 验证集预测vs实际
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'验证集: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. 残差分布（训练集）
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'训练集残差分布 (均值={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. 残差分布（验证集）
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'验证集残差分布 (均值={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. 主函数 ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    主函数
    
    参数:
    - outlier_method: 'remove' 或 'clip'，选择异常值处理方法
    - iqr_multiplier: IQR倍数，用于确定异常值边界
    - use_ensemble: 是否使用集成模型
    """
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    print(f"异常值处理方法: {outlier_method}")
    print(f"IQR倍数: {iqr_multiplier}")
    print(f"使用集成模型: {use_ensemble}")
    print("="*50)
    
    # 1. 加载数据（使用删除异常值方法）
    X, y, column_names = load_and_preprocess_data(
        "填补结果_Earthquake_包含L列_平衡版.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. 数据标准化
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\n数据集划分:")
    print(f"训练集大小: {X_train.shape}")
    print(f"验证集大小: {X_val.shape}")
    print(f"特征数量: {X_train.shape[1]}")
    
    # 4. 选择模型类型
    if use_ensemble:
        print("\n使用优化的集成GAN模型...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\n使用单个优化的GAN模型...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. 模型诊断
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. 可视化结果
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. 输出最终结果
    print("\n" + "="*50)
    print("最终结果总结")
    print("="*50)
    print(f"异常值处理方法: {outlier_method}")
    print(f"处理后数据量: {X.shape[0]} 行")
    print(f"\n原始尺度R²分数:")
    print(f"  训练集: {results['train_r2']:.4f}")
    print(f"  验证集: {results['val_r2']:.4f}")
    print(f"  差异: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. 保存预测结果到CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 预测结果已保存到 train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 完全达标！")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  模型性能优秀，无欠拟合或过拟合问题")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ 部分达标")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\n建议：")
        print("  1. 进一步检查训练数据质量")
        print("  2. 考虑数据清洗或特征工程")
        print("  3. 调整模型架构或训练策略")
    else:
        print(f"\n❌ 未达标")
        print(f"  验证集R² = {results['val_r2']:.4f} < 0.8")
        print("\n建议：")
        print("  1. 增加模型复杂度")
        print("  2. 收集更多训练数据")
        print("  3. 尝试其他模型架构")
    
    return model, results

# 运行主程序
if __name__ == "__main__":
    # 可以调整参数：
    # outlier_method: 'remove'（删除）或 'clip'（截断）
    # iqr_multiplier: 1.5（严格）、3（标准）、5（宽松）
    # use_ensemble: True（集成模型）或 False（单模型）
    
    model, results = main(
        outlier_method='clip',  # 使用删除法
        iqr_multiplier=1.5,         # 使用3倍IQR
        use_ensemble=True         # 使用集成模型
    )
    
    print("\n模型训练完成！结果已保存为 'gan_outlier_removal_results.png'")

In [ ]:
#mixed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. 数据加载和预处理（修改为删除异常值） ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    加载并预处理数据
    
    参数:
    - file_path: 数据文件路径
    - outlier_method: 异常值处理方法 ('remove'=删除, 'clip'=截断)
    - iqr_multiplier: IQR倍数，用于确定异常值边界（默认3倍，更宽松）
    """
    print("正在加载数据...")
    data = pd.read_csv(file_path)
    print(f"原始数据形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 检查是否存在L列
    if 'L(m)' not in data.columns:
        raise ValueError("数据中未找到'L(m)'列，请检查数据格式")
    
    # 处理缺失值
    print(f"\n缺失值统计:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"删除缺失值: {initial_shape[0] - data.shape[0]} 行")
    print(f"删除缺失值后数据形状: {data.shape}")
    
    # 保存处理前的数据用于对比
    data_before_outlier = data.copy()
    
    # 异常值检测
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\n异常值检测（使用{iqr_multiplier}倍IQR方法）:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== 删除异常值方法 ==========
        # 创建一个布尔掩码，标记所有正常值
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 找出异常值
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # 统计异常值
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\n列 '{col}':")
                print(f"  范围: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  正常值边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  异常值数量: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - 低于下界: {n_outliers_low} 个")
                print(f"    - 高于上界: {n_outliers_high} 个")
            
            # 更新掩码（只保留所有特征都正常的行）
            mask = mask & ~col_outliers
        
        # 应用掩码删除异常值
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"异常值删除总结:")
        print(f"  原始数据行数: {len(data)}")
        print(f"  删除异常值行数: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  剩余数据行数: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== 截断异常值方法（原方法） ==========
        print("\n使用截断法处理异常值...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 统计异常值数量
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\n列 '{col}':")
                print(f"  - 低于下界的值: {outliers_low} 个")
                print(f"  - 高于上界的值: {outliers_high} 个")
                print(f"  - 边界: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # 进行截断
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\n总共截断了 {modified_count} 个异常值")
    
    print(f"\n最终数据形状: {data.shape}")
    
    # 显示处理前后的统计对比
    print("\n" + "="*50)
    print("数据统计信息对比:")
    print("="*50)
    print("\n处理前统计信息:")
    print(data_before_outlier.describe().round(2))
    print("\n处理后统计信息:")
    print(data.describe().round(2))
    
    # 分离特征和目标
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. 增强的GAN模型定义 ====================
class FeatureAttention(nn.Module):
    """特征注意力机制"""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """增强的生成器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # 输入投影层
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # 隐藏层
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # 输出层
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """增强的判别器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. 优化的GAN训练类 ====================
class OptimizedGANRegressor:
    """优化的GAN回归模型"""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # 初始化网络
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # 优化器
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # 损失函数
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # 训练历史
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # 早停参数
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """计算梯度惩罚"""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """单步训练"""
        batch_size = real_x.size(0)
        
        # 动态调整标签平滑
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # 训练判别器
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # 训练生成器
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """训练模型"""
        self.generator.train()
        self.discriminator.train()
        
        # 创建数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # 训练循环
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # 学习率调度
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # 评估
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # 记录历史
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # 保存最佳模型
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # 早停
            if self.patience_counter >= patience and epoch > 100:
                print(f"\n早停触发！")
                print(f"最佳训练R2: {self.best_train_r2:.4f}")
                print(f"最佳验证R2: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # 如果训练集R2仍然很低，继续训练
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\n训练集R2仍然较低({train_r2:.4f})，继续训练...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """预测"""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """评估模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. 集成学习包装器 ====================
class OptimizedEnsembleGAN:
    """优化的集成GAN模型"""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """训练多个模型"""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"训练模型 {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # 设置不同的随机种子
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # 创建并训练模型
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # 训练模型
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # 评估
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\n模型 {i+1} 结果:")
            print(f"  训练集R2: {train_score:.4f}")
            print(f"  验证集R2: {val_score:.4f}")
        
        # 计算加权平均的权重
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\n模型权重: {self.weights}")
    
    def predict(self, X):
        """集成预测"""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """评估集成模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. 模型诊断和可视化函数 ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """诊断模型性能"""
    print("\n" + "="*50)
    print("模型诊断报告")
    print("="*50)
    
    # 训练集性能
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\n训练集性能（原始尺度）:")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # 验证集性能
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\n验证集性能（原始尺度）:")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # 诊断结论
    print("\n" + "-"*50)
    print("诊断结论:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ 异常情况：训练集欠拟合，但验证集表现良好")
            print("  可能原因：训练集包含更多噪声或异常值")
        else:
            print("⚠️ 模型存在欠拟合问题")
            print(f"  - 训练集R²较低: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ 模型存在过拟合")
        print(f"  - 训练集与验证集R²差异: {r2_diff:.4f}")
    else:
        print("✅ 模型表现良好")
        print(f"  - 训练集R²: {train_r2:.4f}")
        print(f"  - 验证集R²: {val_r2:.4f}")
        print(f"  - 差异: {r2_diff:.4f}")
    
    # 检查是否达到目标
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 目标达成！")
        print(f"  - 验证集R² >= 0.8: {val_r2:.4f}")
        print(f"  - 训练集R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ 验证集达标(R²={val_r2:.4f})，但训练集仍需改进(R²={train_r2:.4f})")
    else:
        print(f"\n❌ 未达到目标（验证集R²={val_r2:.4f} < 0.8）")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """绘制结果图表"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 处理history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. 损失曲线
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练损失曲线')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R²曲线
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R²分数变化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 训练集预测vs实际
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'训练集: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. 验证集预测vs实际
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'验证集: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. 残差分布（训练集）
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'训练集残差分布 (均值={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. 残差分布（验证集）
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'验证集残差分布 (均值={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. 主函数 ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    主函数
    
    参数:
    - outlier_method: 'remove' 或 'clip'，选择异常值处理方法
    - iqr_multiplier: IQR倍数，用于确定异常值边界
    - use_ensemble: 是否使用集成模型
    """
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    print(f"异常值处理方法: {outlier_method}")
    print(f"IQR倍数: {iqr_multiplier}")
    print(f"使用集成模型: {use_ensemble}")
    print("="*50)
    
    # 1. 加载数据（使用删除异常值方法）
    X, y, column_names = load_and_preprocess_data(
        "填补结果_Mixed_包含L列_平衡版.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. 数据标准化
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\n数据集划分:")
    print(f"训练集大小: {X_train.shape}")
    print(f"验证集大小: {X_val.shape}")
    print(f"特征数量: {X_train.shape[1]}")
    
    # 4. 选择模型类型
    if use_ensemble:
        print("\n使用优化的集成GAN模型...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\n使用单个优化的GAN模型...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. 模型诊断
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. 可视化结果
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. 输出最终结果
    print("\n" + "="*50)
    print("最终结果总结")
    print("="*50)
    print(f"异常值处理方法: {outlier_method}")
    print(f"处理后数据量: {X.shape[0]} 行")
    print(f"\n原始尺度R²分数:")
    print(f"  训练集: {results['train_r2']:.4f}")
    print(f"  验证集: {results['val_r2']:.4f}")
    print(f"  差异: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. 保存预测结果到CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 预测结果已保存到 train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 完全达标！")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  模型性能优秀，无欠拟合或过拟合问题")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ 部分达标")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\n建议：")
        print("  1. 进一步检查训练数据质量")
        print("  2. 考虑数据清洗或特征工程")
        print("  3. 调整模型架构或训练策略")
    else:
        print(f"\n❌ 未达标")
        print(f"  验证集R² = {results['val_r2']:.4f} < 0.8")
        print("\n建议：")
        print("  1. 增加模型复杂度")
        print("  2. 收集更多训练数据")
        print("  3. 尝试其他模型架构")
    
    return model, results

# 运行主程序
if __name__ == "__main__":
    # 可以调整参数：
    # outlier_method: 'remove'（删除）或 'clip'（截断）
    # iqr_multiplier: 1.5（严格）、3（标准）、5（宽松）
    # use_ensemble: True（集成模型）或 False（单模型）
    
    model, results = main(
        outlier_method='clip',  # 使用删除法
        iqr_multiplier=1.5,         # 使用3倍IQR
        use_ensemble=True         # 使用集成模型
    )
    
    print("\n模型训练完成！结果已保存为 'gan_outlier_removal_results.png'")

In [ ]:
#others
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. 数据加载和预处理（修改为删除异常值） ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    加载并预处理数据
    
    参数:
    - file_path: 数据文件路径
    - outlier_method: 异常值处理方法 ('remove'=删除, 'clip'=截断)
    - iqr_multiplier: IQR倍数，用于确定异常值边界（默认3倍，更宽松）
    """
    print("正在加载数据...")
    data = pd.read_csv(file_path)
    print(f"原始数据形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 检查是否存在L列
    if 'L(m)' not in data.columns:
        raise ValueError("数据中未找到'L(m)'列，请检查数据格式")
    
    # 处理缺失值
    print(f"\n缺失值统计:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"删除缺失值: {initial_shape[0] - data.shape[0]} 行")
    print(f"删除缺失值后数据形状: {data.shape}")
    
    # 保存处理前的数据用于对比
    data_before_outlier = data.copy()
    
    # 异常值检测
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\n异常值检测（使用{iqr_multiplier}倍IQR方法）:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== 删除异常值方法 ==========
        # 创建一个布尔掩码，标记所有正常值
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 找出异常值
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # 统计异常值
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\n列 '{col}':")
                print(f"  范围: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  正常值边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  异常值数量: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - 低于下界: {n_outliers_low} 个")
                print(f"    - 高于上界: {n_outliers_high} 个")
            
            # 更新掩码（只保留所有特征都正常的行）
            mask = mask & ~col_outliers
        
        # 应用掩码删除异常值
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"异常值删除总结:")
        print(f"  原始数据行数: {len(data)}")
        print(f"  删除异常值行数: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  剩余数据行数: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== 截断异常值方法（原方法） ==========
        print("\n使用截断法处理异常值...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 统计异常值数量
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\n列 '{col}':")
                print(f"  - 低于下界的值: {outliers_low} 个")
                print(f"  - 高于上界的值: {outliers_high} 个")
                print(f"  - 边界: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # 进行截断
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\n总共截断了 {modified_count} 个异常值")
    
    print(f"\n最终数据形状: {data.shape}")
    
    # 显示处理前后的统计对比
    print("\n" + "="*50)
    print("数据统计信息对比:")
    print("="*50)
    print("\n处理前统计信息:")
    print(data_before_outlier.describe().round(2))
    print("\n处理后统计信息:")
    print(data.describe().round(2))
    
    # 分离特征和目标
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. 增强的GAN模型定义 ====================
class FeatureAttention(nn.Module):
    """特征注意力机制"""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """增强的生成器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # 输入投影层
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # 隐藏层
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # 输出层
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """增强的判别器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. 优化的GAN训练类 ====================
class OptimizedGANRegressor:
    """优化的GAN回归模型"""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # 初始化网络
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # 优化器
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # 损失函数
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # 训练历史
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # 早停参数
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """计算梯度惩罚"""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """单步训练"""
        batch_size = real_x.size(0)
        
        # 动态调整标签平滑
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # 训练判别器
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # 训练生成器
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """训练模型"""
        self.generator.train()
        self.discriminator.train()
        
        # 创建数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # 训练循环
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # 学习率调度
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # 评估
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # 记录历史
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # 保存最佳模型
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # 早停
            if self.patience_counter >= patience and epoch > 100:
                print(f"\n早停触发！")
                print(f"最佳训练R2: {self.best_train_r2:.4f}")
                print(f"最佳验证R2: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # 如果训练集R2仍然很低，继续训练
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\n训练集R2仍然较低({train_r2:.4f})，继续训练...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """预测"""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        # ✅ 保证预测值为正数
        predictions = np.maximum(predictions, 0)
        return predictions
    
    def evaluate(self, X, y):
        """评估模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. 集成学习包装器 ====================
class OptimizedEnsembleGAN:
    """优化的集成GAN模型"""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """训练多个模型"""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"训练模型 {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # 设置不同的随机种子
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # 创建并训练模型
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # 训练模型
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # 评估
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\n模型 {i+1} 结果:")
            print(f"  训练集R2: {train_score:.4f}")
            print(f"  验证集R2: {val_score:.4f}")
        
        # 计算加权平均的权重
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\n模型权重: {self.weights}")
    
    def predict(self, X):
        """集成预测"""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        # ✅ 保证预测值为正数
        weighted_pred = np.maximum(weighted_pred, 0)
        return weighted_pred
    
    def evaluate(self, X, y):
        """评估集成模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. 模型诊断和可视化函数 ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """诊断模型性能"""
    print("\n" + "="*50)
    print("模型诊断报告")
    print("="*50)
    
    # 训练集性能
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\n训练集性能（原始尺度）:")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # 验证集性能
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\n验证集性能（原始尺度）:")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # 诊断结论
    print("\n" + "-"*50)
    print("诊断结论:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ 异常情况：训练集欠拟合，但验证集表现良好")
            print("  可能原因：训练集包含更多噪声或异常值")
        else:
            print("⚠️ 模型存在欠拟合问题")
            print(f"  - 训练集R²较低: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ 模型存在过拟合")
        print(f"  - 训练集与验证集R²差异: {r2_diff:.4f}")
    else:
        print("✅ 模型表现良好")
        print(f"  - 训练集R²: {train_r2:.4f}")
        print(f"  - 验证集R²: {val_r2:.4f}")
        print(f"  - 差异: {r2_diff:.4f}")
    
    # 检查是否达到目标
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 目标达成！")
        print(f"  - 验证集R² >= 0.8: {val_r2:.4f}")
        print(f"  - 训练集R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ 验证集达标(R²={val_r2:.4f})，但训练集仍需改进(R²={train_r2:.4f})")
    else:
        print(f"\n❌ 未达到目标（验证集R²={val_r2:.4f} < 0.8）")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """绘制结果图表"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 处理history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. 损失曲线
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练损失曲线')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R²曲线
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R²分数变化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 训练集预测vs实际
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'训练集: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. 验证集预测vs实际
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'验证集: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. 残差分布（训练集）
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'训练集残差分布 (均值={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. 残差分布（验证集）
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'验证集残差分布 (均值={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. 主函数 ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    主函数
    
    参数:
    - outlier_method: 'remove' 或 'clip'，选择异常值处理方法
    - iqr_multiplier: IQR倍数，用于确定异常值边界
    - use_ensemble: 是否使用集成模型
    """
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    print(f"异常值处理方法: {outlier_method}")
    print(f"IQR倍数: {iqr_multiplier}")
    print(f"使用集成模型: {use_ensemble}")
    print("="*50)
    
    # 1. 加载数据（使用删除异常值方法）
    X, y, column_names = load_and_preprocess_data(
        "填补结果_Others_包含L列_平衡版.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. 数据标准化
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\n数据集划分:")
    print(f"训练集大小: {X_train.shape}")
    print(f"验证集大小: {X_val.shape}")
    print(f"特征数量: {X_train.shape[1]}")
    
    # 4. 选择模型类型
    if use_ensemble:
        print("\n使用优化的集成GAN模型...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\n使用单个优化的GAN模型...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. 模型诊断
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. 可视化结果
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. 输出最终结果
    print("\n" + "="*50)
    print("最终结果总结")
    print("="*50)
    print(f"异常值处理方法: {outlier_method}")
    print(f"处理后数据量: {X.shape[0]} 行")
    print(f"\n原始尺度R²分数:")
    print(f"  训练集: {results['train_r2']:.4f}")
    print(f"  验证集: {results['val_r2']:.4f}")
    print(f"  差异: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. 保存预测结果到CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 预测结果已保存到 train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 完全达标！")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  模型性能优秀，无欠拟合或过拟合问题")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ 部分达标")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\n建议：")
        print("  1. 进一步检查训练数据质量")
        print("  2. 考虑数据清洗或特征工程")
        print("  3. 调整模型架构或训练策略")
    else:
        print(f"\n❌ 未达标")
        print(f"  验证集R² = {results['val_r2']:.4f} < 0.8")
        print("\n建议：")
        print("  1. 增加模型复杂度")
        print("  2. 收集更多训练数据")
        print("  3. 尝试其他模型架构")
    
    return model, results

# 运行主程序
if __name__ == "__main__":
    # 可以调整参数：
    # outlier_method: 'remove'（删除）或 'clip'（截断）
    # iqr_multiplier: 1.5（严格）、3（标准）、5（宽松）
    # use_ensemble: True（集成模型）或 False（单模型）
    
    model, results = main(
        outlier_method='clip',  # 使用删除法
        iqr_multiplier=1.5,         # 使用3倍IQR
        use_ensemble=True         # 使用集成模型
    )
    
    print("\n模型训练完成！结果已保存为 'gan_outlier_removal_results.png'")

In [ ]:
#Rainfall
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. 数据加载和预处理（修改为删除异常值） ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    加载并预处理数据
    
    参数:
    - file_path: 数据文件路径
    - outlier_method: 异常值处理方法 ('remove'=删除, 'clip'=截断)
    - iqr_multiplier: IQR倍数，用于确定异常值边界（默认3倍，更宽松）
    """
    print("正在加载数据...")
    data = pd.read_csv(file_path)
    print(f"原始数据形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 检查是否存在L列
    if 'L(m)' not in data.columns:
        raise ValueError("数据中未找到'L(m)'列，请检查数据格式")
    
    # 处理缺失值
    print(f"\n缺失值统计:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"删除缺失值: {initial_shape[0] - data.shape[0]} 行")
    print(f"删除缺失值后数据形状: {data.shape}")
    
    # 保存处理前的数据用于对比
    data_before_outlier = data.copy()
    
    # 异常值检测
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\n异常值检测（使用{iqr_multiplier}倍IQR方法）:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== 删除异常值方法 ==========
        # 创建一个布尔掩码，标记所有正常值
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 找出异常值
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # 统计异常值
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\n列 '{col}':")
                print(f"  范围: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  正常值边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  异常值数量: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - 低于下界: {n_outliers_low} 个")
                print(f"    - 高于上界: {n_outliers_high} 个")
            
            # 更新掩码（只保留所有特征都正常的行）
            mask = mask & ~col_outliers
        
        # 应用掩码删除异常值
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"异常值删除总结:")
        print(f"  原始数据行数: {len(data)}")
        print(f"  删除异常值行数: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  剩余数据行数: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== 截断异常值方法（原方法） ==========
        print("\n使用截断法处理异常值...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 统计异常值数量
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\n列 '{col}':")
                print(f"  - 低于下界的值: {outliers_low} 个")
                print(f"  - 高于上界的值: {outliers_high} 个")
                print(f"  - 边界: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # 进行截断
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\n总共截断了 {modified_count} 个异常值")
    
    print(f"\n最终数据形状: {data.shape}")
    
    # 显示处理前后的统计对比
    print("\n" + "="*50)
    print("数据统计信息对比:")
    print("="*50)
    print("\n处理前统计信息:")
    print(data_before_outlier.describe().round(2))
    print("\n处理后统计信息:")
    print(data.describe().round(2))
    
    # 分离特征和目标
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. 增强的GAN模型定义 ====================
class FeatureAttention(nn.Module):
    """特征注意力机制"""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """增强的生成器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # 输入投影层
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # 隐藏层
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # 输出层
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """增强的判别器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. 优化的GAN训练类 ====================
class OptimizedGANRegressor:
    """优化的GAN回归模型"""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # 初始化网络
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # 优化器
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # 损失函数
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # 训练历史
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # 早停参数
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """计算梯度惩罚"""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """单步训练"""
        batch_size = real_x.size(0)
        
        # 动态调整标签平滑
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # 训练判别器
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # 训练生成器
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """训练模型"""
        self.generator.train()
        self.discriminator.train()
        
        # 创建数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # 训练循环
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # 学习率调度
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # 评估
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # 记录历史
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # 保存最佳模型
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # 早停
            if self.patience_counter >= patience and epoch > 100:
                print(f"\n早停触发！")
                print(f"最佳训练R2: {self.best_train_r2:.4f}")
                print(f"最佳验证R2: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # 如果训练集R2仍然很低，继续训练
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\n训练集R2仍然较低({train_r2:.4f})，继续训练...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """预测"""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """评估模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. 集成学习包装器 ====================
class OptimizedEnsembleGAN:
    """优化的集成GAN模型"""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """训练多个模型"""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"训练模型 {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # 设置不同的随机种子
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # 创建并训练模型
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # 训练模型
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # 评估
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\n模型 {i+1} 结果:")
            print(f"  训练集R2: {train_score:.4f}")
            print(f"  验证集R2: {val_score:.4f}")
        
        # 计算加权平均的权重
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\n模型权重: {self.weights}")
    
    def predict(self, X):
        """集成预测"""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """评估集成模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. 模型诊断和可视化函数 ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """诊断模型性能"""
    print("\n" + "="*50)
    print("模型诊断报告")
    print("="*50)
    
    # 训练集性能
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\n训练集性能（原始尺度）:")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # 验证集性能
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\n验证集性能（原始尺度）:")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # 诊断结论
    print("\n" + "-"*50)
    print("诊断结论:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ 异常情况：训练集欠拟合，但验证集表现良好")
            print("  可能原因：训练集包含更多噪声或异常值")
        else:
            print("⚠️ 模型存在欠拟合问题")
            print(f"  - 训练集R²较低: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ 模型存在过拟合")
        print(f"  - 训练集与验证集R²差异: {r2_diff:.4f}")
    else:
        print("✅ 模型表现良好")
        print(f"  - 训练集R²: {train_r2:.4f}")
        print(f"  - 验证集R²: {val_r2:.4f}")
        print(f"  - 差异: {r2_diff:.4f}")
    
    # 检查是否达到目标
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 目标达成！")
        print(f"  - 验证集R² >= 0.8: {val_r2:.4f}")
        print(f"  - 训练集R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ 验证集达标(R²={val_r2:.4f})，但训练集仍需改进(R²={train_r2:.4f})")
    else:
        print(f"\n❌ 未达到目标（验证集R²={val_r2:.4f} < 0.8）")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """绘制结果图表"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 处理history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. 损失曲线
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练损失曲线')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R²曲线
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R²分数变化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 训练集预测vs实际
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'训练集: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. 验证集预测vs实际
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'验证集: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. 残差分布（训练集）
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'训练集残差分布 (均值={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. 残差分布（验证集）
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'验证集残差分布 (均值={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. 主函数 ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    主函数
    
    参数:
    - outlier_method: 'remove' 或 'clip'，选择异常值处理方法
    - iqr_multiplier: IQR倍数，用于确定异常值边界
    - use_ensemble: 是否使用集成模型
    """
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    print(f"异常值处理方法: {outlier_method}")
    print(f"IQR倍数: {iqr_multiplier}")
    print(f"使用集成模型: {use_ensemble}")
    print("="*50)
    
    # 1. 加载数据（使用删除异常值方法）
    X, y, column_names = load_and_preprocess_data(
        "填补结果_Rainfall_包含L列.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. 数据标准化
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\n数据集划分:")
    print(f"训练集大小: {X_train.shape}")
    print(f"验证集大小: {X_val.shape}")
    print(f"特征数量: {X_train.shape[1]}")
    
    # 4. 选择模型类型
    if use_ensemble:
        print("\n使用优化的集成GAN模型...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\n使用单个优化的GAN模型...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. 模型诊断
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. 可视化结果
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. 输出最终结果
    print("\n" + "="*50)
    print("最终结果总结")
    print("="*50)
    print(f"异常值处理方法: {outlier_method}")
    print(f"处理后数据量: {X.shape[0]} 行")
    print(f"\n原始尺度R²分数:")
    print(f"  训练集: {results['train_r2']:.4f}")
    print(f"  验证集: {results['val_r2']:.4f}")
    print(f"  差异: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. 保存预测结果到CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 预测结果已保存到 train_val_predictions.csv")

    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 完全达标！")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  模型性能优秀，无欠拟合或过拟合问题")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ 部分达标")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\n建议：")
        print("  1. 进一步检查训练数据质量")
        print("  2. 考虑数据清洗或特征工程")
        print("  3. 调整模型架构或训练策略")
    else:
        print(f"\n❌ 未达标")
        print(f"  验证集R² = {results['val_r2']:.4f} < 0.8")
        print("\n建议：")
        print("  1. 增加模型复杂度")
        print("  2. 收集更多训练数据")
        print("  3. 尝试其他模型架构")
    
    return model, results

# 运行主程序
if __name__ == "__main__":
    # 可以调整参数：
    # outlier_method: 'remove'（删除）或 'clip'（截断）
    # iqr_multiplier: 1.5（严格）、3（标准）、5（宽松）
    # use_ensemble: True（集成模型）或 False（单模型）
    
    model, results = main(
        outlier_method='clip',  # 使用删除法
        iqr_multiplier=1.5,         # 使用3倍IQR
        use_ensemble=True         # 使用集成模型
    )
    
    print("\n模型训练完成！结果已保存为 'gan_outlier_removal_results.png'")

In [ ]:
#rock
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. 数据加载和预处理（修改为删除异常值） ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    加载并预处理数据
    
    参数:
    - file_path: 数据文件路径
    - outlier_method: 异常值处理方法 ('remove'=删除, 'clip'=截断)
    - iqr_multiplier: IQR倍数，用于确定异常值边界（默认3倍，更宽松）
    """
    print("正在加载数据...")
    data = pd.read_csv(file_path)
    print(f"原始数据形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 检查是否存在L列
    if 'L(m)' not in data.columns:
        raise ValueError("数据中未找到'L(m)'列，请检查数据格式")
    
    # 处理缺失值
    print(f"\n缺失值统计:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"删除缺失值: {initial_shape[0] - data.shape[0]} 行")
    print(f"删除缺失值后数据形状: {data.shape}")
    
    # 保存处理前的数据用于对比
    data_before_outlier = data.copy()
    
    # 异常值检测
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\n异常值检测（使用{iqr_multiplier}倍IQR方法）:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== 删除异常值方法 ==========
        # 创建一个布尔掩码，标记所有正常值
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 找出异常值
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # 统计异常值
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\n列 '{col}':")
                print(f"  范围: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  正常值边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  异常值数量: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - 低于下界: {n_outliers_low} 个")
                print(f"    - 高于上界: {n_outliers_high} 个")
            
            # 更新掩码（只保留所有特征都正常的行）
            mask = mask & ~col_outliers
        
        # 应用掩码删除异常值
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"异常值删除总结:")
        print(f"  原始数据行数: {len(data)}")
        print(f"  删除异常值行数: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  剩余数据行数: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== 截断异常值方法（原方法） ==========
        print("\n使用截断法处理异常值...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 统计异常值数量
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\n列 '{col}':")
                print(f"  - 低于下界的值: {outliers_low} 个")
                print(f"  - 高于上界的值: {outliers_high} 个")
                print(f"  - 边界: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # 进行截断
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\n总共截断了 {modified_count} 个异常值")
    
    print(f"\n最终数据形状: {data.shape}")
    
    # 显示处理前后的统计对比
    print("\n" + "="*50)
    print("数据统计信息对比:")
    print("="*50)
    print("\n处理前统计信息:")
    print(data_before_outlier.describe().round(2))
    print("\n处理后统计信息:")
    print(data.describe().round(2))
    
    # 分离特征和目标
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. 增强的GAN模型定义 ====================
class FeatureAttention(nn.Module):
    """特征注意力机制"""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """增强的生成器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # 输入投影层
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # 隐藏层
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # 输出层
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """增强的判别器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. 优化的GAN训练类 ====================
class OptimizedGANRegressor:
    """优化的GAN回归模型"""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # 初始化网络
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # 优化器
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # 损失函数
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # 训练历史
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # 早停参数
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """计算梯度惩罚"""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """单步训练"""
        batch_size = real_x.size(0)
        
        # 动态调整标签平滑
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # 训练判别器
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # 训练生成器
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """训练模型"""
        self.generator.train()
        self.discriminator.train()
        
        # 创建数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # 训练循环
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # 学习率调度
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # 评估
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # 记录历史
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # 保存最佳模型
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # 早停
            if self.patience_counter >= patience and epoch > 100:
                print(f"\n早停触发！")
                print(f"最佳训练R2: {self.best_train_r2:.4f}")
                print(f"最佳验证R2: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # 如果训练集R2仍然很低，继续训练
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\n训练集R2仍然较低({train_r2:.4f})，继续训练...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """预测"""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """评估模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. 集成学习包装器 ====================
class OptimizedEnsembleGAN:
    """优化的集成GAN模型"""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """训练多个模型"""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"训练模型 {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # 设置不同的随机种子
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # 创建并训练模型
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # 训练模型
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # 评估
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\n模型 {i+1} 结果:")
            print(f"  训练集R2: {train_score:.4f}")
            print(f"  验证集R2: {val_score:.4f}")
        
        # 计算加权平均的权重
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\n模型权重: {self.weights}")
    
    def predict(self, X):
        """集成预测"""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """评估集成模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. 模型诊断和可视化函数 ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """诊断模型性能"""
    print("\n" + "="*50)
    print("模型诊断报告")
    print("="*50)
    
    # 训练集性能
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\n训练集性能（原始尺度）:")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # 验证集性能
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\n验证集性能（原始尺度）:")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # 诊断结论
    print("\n" + "-"*50)
    print("诊断结论:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ 异常情况：训练集欠拟合，但验证集表现良好")
            print("  可能原因：训练集包含更多噪声或异常值")
        else:
            print("⚠️ 模型存在欠拟合问题")
            print(f"  - 训练集R²较低: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ 模型存在过拟合")
        print(f"  - 训练集与验证集R²差异: {r2_diff:.4f}")
    else:
        print("✅ 模型表现良好")
        print(f"  - 训练集R²: {train_r2:.4f}")
        print(f"  - 验证集R²: {val_r2:.4f}")
        print(f"  - 差异: {r2_diff:.4f}")
    
    # 检查是否达到目标
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 目标达成！")
        print(f"  - 验证集R² >= 0.8: {val_r2:.4f}")
        print(f"  - 训练集R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ 验证集达标(R²={val_r2:.4f})，但训练集仍需改进(R²={train_r2:.4f})")
    else:
        print(f"\n❌ 未达到目标（验证集R²={val_r2:.4f} < 0.8）")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """绘制结果图表"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 处理history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. 损失曲线
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练损失曲线')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R²曲线
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R²分数变化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 训练集预测vs实际
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'训练集: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. 验证集预测vs实际
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'验证集: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. 残差分布（训练集）
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'训练集残差分布 (均值={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. 残差分布（验证集）
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'验证集残差分布 (均值={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. 主函数 ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    主函数
    
    参数:
    - outlier_method: 'remove' 或 'clip'，选择异常值处理方法
    - iqr_multiplier: IQR倍数，用于确定异常值边界
    - use_ensemble: 是否使用集成模型
    """
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    print(f"异常值处理方法: {outlier_method}")
    print(f"IQR倍数: {iqr_multiplier}")
    print(f"使用集成模型: {use_ensemble}")
    print("="*50)
    
    # 1. 加载数据（使用删除异常值方法）
    X, y, column_names = load_and_preprocess_data(
        "填补结果_Rock_包含L列_平衡版.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. 数据标准化
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\n数据集划分:")
    print(f"训练集大小: {X_train.shape}")
    print(f"验证集大小: {X_val.shape}")
    print(f"特征数量: {X_train.shape[1]}")
    
    # 4. 选择模型类型
    if use_ensemble:
        print("\n使用优化的集成GAN模型...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\n使用单个优化的GAN模型...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. 模型诊断
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. 可视化结果
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. 输出最终结果
    print("\n" + "="*50)
    print("最终结果总结")
    print("="*50)
    print(f"异常值处理方法: {outlier_method}")
    print(f"处理后数据量: {X.shape[0]} 行")
    print(f"\n原始尺度R²分数:")
    print(f"  训练集: {results['train_r2']:.4f}")
    print(f"  验证集: {results['val_r2']:.4f}")
    print(f"  差异: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. 保存预测结果到CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 预测结果已保存到 train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 完全达标！")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  模型性能优秀，无欠拟合或过拟合问题")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ 部分达标")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\n建议：")
        print("  1. 进一步检查训练数据质量")
        print("  2. 考虑数据清洗或特征工程")
        print("  3. 调整模型架构或训练策略")
    else:
        print(f"\n❌ 未达标")
        print(f"  验证集R² = {results['val_r2']:.4f} < 0.8")
        print("\n建议：")
        print("  1. 增加模型复杂度")
        print("  2. 收集更多训练数据")
        print("  3. 尝试其他模型架构")
    
    return model, results

# 运行主程序
if __name__ == "__main__":
    # 可以调整参数：
    # outlier_method: 'remove'（删除）或 'clip'（截断）
    # iqr_multiplier: 1.5（严格）、3（标准）、5（宽松）
    # use_ensemble: True（集成模型）或 False（单模型）
    
    model, results = main(
        outlier_method='clip',  # 使用删除法
        iqr_multiplier=1.5,         # 使用3倍IQR
        use_ensemble=True         # 使用集成模型
    )
    
    print("\n模型训练完成！结果已保存为 'gan_outlier_removal_results.png'")

In [ ]:
#soil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. 数据加载和预处理（修改为删除异常值） ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    加载并预处理数据
    
    参数:
    - file_path: 数据文件路径
    - outlier_method: 异常值处理方法 ('remove'=删除, 'clip'=截断)
    - iqr_multiplier: IQR倍数，用于确定异常值边界（默认3倍，更宽松）
    """
    print("正在加载数据...")
    data = pd.read_csv(file_path)
    print(f"原始数据形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 检查是否存在L列
    if 'L(m)' not in data.columns:
        raise ValueError("数据中未找到'L(m)'列，请检查数据格式")
    
    # 处理缺失值
    print(f"\n缺失值统计:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"删除缺失值: {initial_shape[0] - data.shape[0]} 行")
    print(f"删除缺失值后数据形状: {data.shape}")
    
    # 保存处理前的数据用于对比
    data_before_outlier = data.copy()
    
    # 异常值检测
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\n异常值检测（使用{iqr_multiplier}倍IQR方法）:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== 删除异常值方法 ==========
        # 创建一个布尔掩码，标记所有正常值
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 找出异常值
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # 统计异常值
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\n列 '{col}':")
                print(f"  范围: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  正常值边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  异常值数量: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - 低于下界: {n_outliers_low} 个")
                print(f"    - 高于上界: {n_outliers_high} 个")
            
            # 更新掩码（只保留所有特征都正常的行）
            mask = mask & ~col_outliers
        
        # 应用掩码删除异常值
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"异常值删除总结:")
        print(f"  原始数据行数: {len(data)}")
        print(f"  删除异常值行数: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  剩余数据行数: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== 截断异常值方法（原方法） ==========
        print("\n使用截断法处理异常值...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # 统计异常值数量
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\n列 '{col}':")
                print(f"  - 低于下界的值: {outliers_low} 个")
                print(f"  - 高于上界的值: {outliers_high} 个")
                print(f"  - 边界: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # 进行截断
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\n总共截断了 {modified_count} 个异常值")
    
    print(f"\n最终数据形状: {data.shape}")
    
    # 显示处理前后的统计对比
    print("\n" + "="*50)
    print("数据统计信息对比:")
    print("="*50)
    print("\n处理前统计信息:")
    print(data_before_outlier.describe().round(2))
    print("\n处理后统计信息:")
    print(data.describe().round(2))
    
    # 分离特征和目标
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. 增强的GAN模型定义 ====================
class FeatureAttention(nn.Module):
    """特征注意力机制"""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """增强的生成器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # 输入投影层
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # 隐藏层
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # 输出层
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """增强的判别器网络"""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. 优化的GAN训练类 ====================
class OptimizedGANRegressor:
    """优化的GAN回归模型"""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # 初始化网络
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # 优化器
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # 损失函数
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # 训练历史
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # 早停参数
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """计算梯度惩罚"""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """单步训练"""
        batch_size = real_x.size(0)
        
        # 动态调整标签平滑
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # 训练判别器
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # 训练生成器
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """训练模型"""
        self.generator.train()
        self.discriminator.train()
        
        # 创建数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # 训练循环
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # 学习率调度
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # 评估
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # 记录历史
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # 保存最佳模型
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # 早停
            if self.patience_counter >= patience and epoch > 100:
                print(f"\n早停触发！")
                print(f"最佳训练R2: {self.best_train_r2:.4f}")
                print(f"最佳验证R2: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # 如果训练集R2仍然很低，继续训练
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\n训练集R2仍然较低({train_r2:.4f})，继续训练...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """预测"""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """评估模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. 集成学习包装器 ====================
class OptimizedEnsembleGAN:
    """优化的集成GAN模型"""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """训练多个模型"""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"训练模型 {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # 设置不同的随机种子
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # 创建并训练模型
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # 训练模型
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # 评估
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\n模型 {i+1} 结果:")
            print(f"  训练集R2: {train_score:.4f}")
            print(f"  验证集R2: {val_score:.4f}")
        
        # 计算加权平均的权重
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\n模型权重: {self.weights}")
    
    def predict(self, X):
        """集成预测"""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """评估集成模型"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. 模型诊断和可视化函数 ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """诊断模型性能"""
    print("\n" + "="*50)
    print("模型诊断报告")
    print("="*50)
    
    # 训练集性能
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\n训练集性能（原始尺度）:")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # 验证集性能
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\n验证集性能（原始尺度）:")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # 诊断结论
    print("\n" + "-"*50)
    print("诊断结论:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ 异常情况：训练集欠拟合，但验证集表现良好")
            print("  可能原因：训练集包含更多噪声或异常值")
        else:
            print("⚠️ 模型存在欠拟合问题")
            print(f"  - 训练集R²较低: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ 模型存在过拟合")
        print(f"  - 训练集与验证集R²差异: {r2_diff:.4f}")
    else:
        print("✅ 模型表现良好")
        print(f"  - 训练集R²: {train_r2:.4f}")
        print(f"  - 验证集R²: {val_r2:.4f}")
        print(f"  - 差异: {r2_diff:.4f}")
    
    # 检查是否达到目标
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 目标达成！")
        print(f"  - 验证集R² >= 0.8: {val_r2:.4f}")
        print(f"  - 训练集R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ 验证集达标(R²={val_r2:.4f})，但训练集仍需改进(R²={train_r2:.4f})")
    else:
        print(f"\n❌ 未达到目标（验证集R²={val_r2:.4f} < 0.8）")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """绘制结果图表"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 处理history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. 损失曲线
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练损失曲线')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R²曲线
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R²分数变化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 训练集预测vs实际
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'训练集: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. 验证集预测vs实际
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('实际值')
    ax.set_ylabel('预测值')
    ax.set_title(f'验证集: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. 残差分布（训练集）
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'训练集残差分布 (均值={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. 残差分布（验证集）
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('残差')
    ax.set_ylabel('频数')
    ax.set_title(f'验证集残差分布 (均值={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. 主函数 ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    主函数
    
    参数:
    - outlier_method: 'remove' 或 'clip'，选择异常值处理方法
    - iqr_multiplier: IQR倍数，用于确定异常值边界
    - use_ensemble: 是否使用集成模型
    """
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    print(f"异常值处理方法: {outlier_method}")
    print(f"IQR倍数: {iqr_multiplier}")
    print(f"使用集成模型: {use_ensemble}")
    print("="*50)
    
    # 1. 加载数据（使用删除异常值方法）
    X, y, column_names = load_and_preprocess_data(
        "填补结果_Soil_包含L列_平衡版.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. 数据标准化
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\n数据集划分:")
    print(f"训练集大小: {X_train.shape}")
    print(f"验证集大小: {X_val.shape}")
    print(f"特征数量: {X_train.shape[1]}")
    
    # 4. 选择模型类型
    if use_ensemble:
        print("\n使用优化的集成GAN模型...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\n使用单个优化的GAN模型...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. 模型诊断
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. 可视化结果
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. 输出最终结果
    print("\n" + "="*50)
    print("最终结果总结")
    print("="*50)
    print(f"异常值处理方法: {outlier_method}")
    print(f"处理后数据量: {X.shape[0]} 行")
    print(f"\n原始尺度R²分数:")
    print(f"  训练集: {results['train_r2']:.4f}")
    print(f"  验证集: {results['val_r2']:.4f}")
    print(f"  差异: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. 保存预测结果到CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 预测结果已保存到 train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 完全达标！")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  模型性能优秀，无欠拟合或过拟合问题")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ 部分达标")
        print(f"  验证集R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  训练集R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\n建议：")
        print("  1. 进一步检查训练数据质量")
        print("  2. 考虑数据清洗或特征工程")
        print("  3. 调整模型架构或训练策略")
    else:
        print(f"\n❌ 未达标")
        print(f"  验证集R² = {results['val_r2']:.4f} < 0.8")
        print("\n建议：")
        print("  1. 增加模型复杂度")
        print("  2. 收集更多训练数据")
        print("  3. 尝试其他模型架构")
    
    return model, results

# 运行主程序
if __name__ == "__main__":
    # 可以调整参数：
    # outlier_method: 'remove'（删除）或 'clip'（截断）
    # iqr_multiplier: 1.5（严格）、3（标准）、5（宽松）
    # use_ensemble: True（集成模型）或 False（单模型）
    
    model, results = main(
        outlier_method='clip',  # 使用删除法
        iqr_multiplier=1.5,         # 使用3倍IQR
        use_ensemble=True         # 使用集成模型
    )
    
    print("\n模型训练完成！结果已保存为 'gan_outlier_removal_results.png'")